# Combinatória e Probabilidade

  Buscando desenvolver a base para combinatória e probabilidade, foram importadas constantes e funções de outros módulos programados anteriormente.

  Para os cálculos de combinatória, foram implementadas funções para calcular a permutação simples, permutação com repetição, permutação circular, arranjo simples, arranjo com repetição, combinação simples, combinação com repetição, triângulo de pascal, número binomial, número multinomial, número de subconjuntos, número de desarranjos, número de Catalan, número de Stirling de segunda espécie e número de Bell.

  Para a base de probabilidade foi definida as classes "EspacoAmostral", essa é uma representação dos espaços amostrais. Nela, foram definidas as partes que compõem um espaço amostral na classe. Também foram definidas as maneiras com que é interpretado internamente, como ela aparecerá ao ser printada, como aparecerá internamente no código, dentre outras propriedades. Também foram definidas as propriedades dentro de um espaço amostral, sendo a probabilidade de ocorrência de um evento, a probabilidade de união de dois eventos, a esperança e a variança.

  Além disso, para o cálculo de probabilidade, foram definidos algoritmos para calcular a probabilidade condicional, verificar se dois eventos são independentes, para o teorema de Bayes e sua forma completa. Para distribuições, foram feitos algoritmos para diversas formas de distribuição uniforme discreta, distribuição uniforme contínua, distribuiçaõ binomial, distribuição geométrica, distribuição hipergeométrica, distribuição de Poisson, distribuição normal, distribuição exponencial, distribuição qui-quadrado e distribuição t de Student.

  Para cálculos probabilisticos diversos, foram implementados algoritmos de amostragem pseudoaleatória uniforme e amostragem pseudoaleatória normal; além de um algoritmo para a lei dos números grandes.

  Para finalizar, adicionou-se a linha de código "%%writefile combinatoria_e_probabilidade.py" para converter o código em um módulo exportável para ser usado em outros módulos e no uso final. Pelo mesmo motivo, o código teve que ser colocado inteiramente em uma única célula de código, pois o comando usado para isso exporta códigos de apenas uma célula para isso.

In [1]:
%%writefile combinatoria_e_probabilidade.py

from constantes import tolerancia_de_erro, maximo_de_interacoes, pi
from aritmetica import valor_absoluto, fatorial
from funcoes import exponencial, ln, sqrt, gamma, erf
from trigonometria import sin, cos

#Permutação simples
def permutacao_simples(n: int) -> int:
    return fatorial(n)


#Permutação com repetição
def permutacao_com_repeticao(n: int, repeticoes: list) -> int:
    denominador = 1
    for r in repeticoes:
        denominador *= fatorial(r)
    return fatorial(n) // denominador


#Permutação circular
def permutacao_circular(n: int) -> int:
    if n <= 0:
        raise ValueError("Permutação circular requer n >= 1.")
    return fatorial(n - 1)


#Arranjo simples
def arranjo(n: int, p: int) -> int:
    if p > n:
        raise ValueError("Arranjo requer p <= n.")
    if p < 0 or n < 0:
        raise ValueError("n e p devem ser não-negativos.")
    resultado = 1
    for k in range(n, n - p, -1):
        resultado *= k
    return resultado


#Arranjo com repetição
def arranjo_com_repeticao(n: int, p: int) -> int:
    return n ** p


#Combinação simples
def combinacao(n: int, p: int) -> int:
    if p > n:
        raise ValueError("Combinação requer p <= n.")
    if p < 0 or n < 0:
        raise ValueError("n e p devem ser não-negativos.")
    if p == 0 or p == n:
        return 1
    p = min(p, n - p)
    numerador = 1
    denominador = 1
    for k in range(p):
        numerador *= (n - k)
        denominador *= (k + 1)
    return numerador // denominador


#Combinação com repetição
def combinacao_com_repeticao(n: int, p: int) -> int:
    return combinacao(n + p - 1, p)


#Triângulo de Pascal
def triangulo_de_pascal(linhas: int) -> list:
    triangulo = []
    for i in range(linhas):
        linha = [1] * (i + 1)
        for j in range(1, i):
            linha[j] = triangulo[i - 1][j - 1] + triangulo[i - 1][j]
        triangulo.append(linha)
    return triangulo


#Número binomial
def numero_binomial(n: int, k: int) -> int:
    return combinacao(n, k)


#Número multinomial
def numero_multinomial(n: int, partes: list) -> int:
    if sum(partes) != n:
        raise ValueError("A soma das partes deve ser igual a n.")
    denominador = 1
    for p in partes:
        denominador *= fatorial(p)
    return fatorial(n) // denominador


#Número de subconjuntos
def numero_de_subconjuntos(n: int) -> int:
    return 2 ** n


#Número de permutações sem nenhum elemento no lugar original (desarranjos).
def numero_de_desarranjos(n: int) -> int:
    if n == 0:
        return 1
    if n == 1:
        return 0
    anterior_2, anterior_1 = 1, 0
    for k in range(2, n + 1):
        atual = (k - 1) * (anterior_1 + anterior_2)
        anterior_2, anterior_1 = anterior_1, atual
    return anterior_1


#Número de Catalan
def numero_de_catalan(n: int) -> int:
    return combinacao(2 * n, n) // (n + 1)


#Número de maneiras de dividir n elementos em k grupos.
def numero_de_stirling_segunda_especie(n: int, k: int) -> int:
    if k > n:
        return 0
    if k == 0:
        return 1 if n == 0 else 0
    tabela = [[0] * (k + 1) for _ in range(n + 1)]
    tabela[0][0] = 1
    for i in range(1, n + 1):
        for j in range(1, min(i, k) + 1):
            tabela[i][j] = j * tabela[i - 1][j] + tabela[i - 1][j - 1]
    return tabela[n][k]


#Número total de partições de um conjunto.
def numero_de_bell(n: int) -> int:
    return sum(numero_de_stirling_segunda_especie(n, k) for k in range(n + 1))


#Definição de classe para espaços amostrais
class EspacoAmostral:

    def __init__(proprio, eventos: list, probabilidades: list = None):
        if probabilidades is None:
            n = len(eventos)
            probabilidades = [1.0 / n] * n

        if len(eventos) != len(probabilidades):
            raise ValueError("Eventos e probabilidades devem ter o mesmo tamanho.")

        soma = sum(probabilidades)
        if valor_absoluto(soma - 1.0) > tolerancia_de_erro * 1000:
            raise ValueError("As probabilidades devem somar 1.")

        proprio.eventos = list(eventos)
        proprio.probabilidades = list(probabilidades)

    #Propriedades dentro do espaço amostral
    #Probabilidade de ocorrência
    def probabilidade_de(proprio, evento) -> float:
        total = 0.0
        for ev, p in zip(proprio.eventos, proprio.probabilidades):
            if ev == evento:
                total += p
        return total


    # Probabilidade da união de dois eventos
    def probabilidade_uniao(proprio, evento_a, evento_b, p_intersecao: float = 0.0) -> float:
        p_a = proprio.probabilidade_de(evento_a)
        p_b = proprio.probabilidade_de(evento_b)
        if evento_a == evento_b:
            return p_a
        return p_a + p_b - p_intersecao


    #Esperança
    def esperanca(proprio, funcao_valor) -> float:
        return sum(funcao_valor(ev) * p for ev, p in zip(proprio.eventos, proprio.probabilidades))


    #Variança
    def variancia(proprio, funcao_valor) -> float:
        media = proprio.esperanca(funcao_valor)
        return sum(p * (funcao_valor(ev) - media) ** 2 for ev, p in zip(proprio.eventos, proprio.probabilidades))


#Probabilidade condicional
def probabilidade_condicional(p_a_intersecao_b: float, p_b: float) -> float:
    if p_b < tolerancia_de_erro:
        raise ZeroDivisionError("P(B) não pode ser zero para probabilidade condicional.")
    return p_a_intersecao_b / p_b


#Verificação se dois eventos são independentes
def eh_independente(p_a: float, p_b: float, p_a_intersecao_b: float, tol: float = None) -> bool:
    if tol is None:
        tol = tolerancia_de_erro * 1000
    return valor_absoluto(p_a_intersecao_b - p_a * p_b) < tol


#Teorema de Bayes
def teorema_de_bayes(p_b_dado_a: float, p_a: float, p_b: float) -> float:
    if p_b < tolerancia_de_erro:
        raise ZeroDivisionError("P(B) não pode ser zero no Teorema de Bayes.")
    return (p_b_dado_a * p_a) / p_b


#Teorema de Bayes completo (para vários eventos)
def teorema_de_bayes_completo(p_b_dado_a: list, p_a: list) -> list:
    if abs(sum(p_a) - 1) > tolerancia_de_erro * 1000:
        raise ValueError("As probabilidades em p_a devem somar 1.")

    if len(p_b_dado_a) != len(p_a):
        raise ValueError("p_b_dado_a e p_a devem ter o mesmo tamanho.")

    p_b = sum(pb * pa for pb, pa in zip(p_b_dado_a, p_a))
    if p_b < tolerancia_de_erro:
        raise ZeroDivisionError("P(B) total ≈ 0 no Teorema de Bayes.")

    return [(pb * pa) / p_b for pb, pa in zip(p_b_dado_a, p_a)]


#Distribuição uniforme discreta
#Distribuição uniforme discreta - função de probabilidade
def distribuicao_uniforme_discreta_pmf(a: int, b: int, x: int) -> float:
    if b < a:
        raise ValueError("Intervalo inválido")
    if x < a or x > b:
        return 0.0
    return 1.0 / (b - a + 1)


#Distribuição uniforme discreta - média
def distribuicao_uniforme_discreta_media(a: int, b: int) -> float:
    return (a + b) / 2.0


#Distribuição uniforme discreta - variância
def distribuicao_uniforme_discreta_variancia(a: int, b: int) -> float:
    n = b - a + 1
    return (n * n - 1) / 12.0


#Distribuição uniforme contínua
#Distribuição uniforme contínua - densidade de probabilidade
def distribuicao_uniforme_continua_pdf(a: float, b: float, x: float) -> float:
    if a >= b:
        raise ValueError("b deve ser maior que a")
    if x < a or x > b:
        return 0.0
    return 1.0 / (b - a)


#Distribuição uniforme contínua - função de distribuição acumulada
def distribuicao_uniforme_continua_cdf(a: float, b: float, x: float) -> float:  
    if a >= b:
        raise ValueError("b deve ser maior que a.")
    if x < a:
        return 0.0
    if x > b:
        return 1.0
    return (x - a) / (b - a)


#Distribuição uniforme contínua - média
def distribuicao_uniforme_continua_media(a: float, b: float) -> float:
    return (a + b) / 2.0


#Distribuição uniforme contínua - variância
def distribuicao_uniforme_continua_variancia(a: float, b: float) -> float:
    return (b - a) ** 2 / 12.0


#Distribuição binomial
#Distribuição binomial - função de probabilidade
def distribuicao_binomial_pmf(n: int, p: float, k: int) -> float:
    if not (0.0 <= p <= 1.0):
        raise ValueError("p deve estar em [0, 1].")
    if k < 0 or k > n:
        return 0.0
    return combinacao(n, k) * (p ** k) * ((1.0 - p) ** (n - k))


#Distribuição binomial - função de distribuição acumulada
def distribuicao_binomial_cdf(n: int, p: float, k: int) -> float:
    return sum(distribuicao_binomial_pmf(n, p, i) for i in range(0, k + 1))


#Distribuição binomial - média
def distribuicao_binomial_media(n: int, p: float) -> float:
    return n * p


#Distribuição binomial - variância
def distribuicao_binomial_variancia(n: int, p: float) -> float:
    return n * p * (1.0 - p)


#Distribuição geométrica
#Distribuição geométrica - função de probabilidade
def distribuicao_geometrica_pmf(p: float, k: int) -> float:
    if not (0.0 < p <= 1.0):
        raise ValueError("p deve estar em (0, 1].")
    if k < 1:
        return 0.0
    return p * ((1.0 - p) ** (k - 1))


#Distribuição geométrica - função de distribuição acumulada
def distribuicao_geometrica_cdf(p: float, k: int) -> float:
    if k < 1:
        return 0.0
    return 1.0 - (1.0 - p) ** k


#Distribuição geométrica - média
def distribuicao_geometrica_media(p: float) -> float:
    return 1.0 / p


#Distribuição geométrica - variância
def distribuicao_geometrica_variancia(p: float) -> float:
    return (1.0 - p) / (p * p)


#Distribuição hipergeométrica
#Distribuição hipergeométrica - função de probabilidade
def distribuicao_hipergeometrica_pmf(N: int, K: int, n: int, k: int) -> float:
    if k < 0 or k > n or k > K or (n - k) > (N - K):
        return 0.0
    return (combinacao(K, k) * combinacao(N - K, n - k)) / combinacao(N, n)


#Distribuição hipergeométrica - média
def distribuicao_hipergeometrica_media(N: int, K: int, n: int) -> float:
    return n * K / N


#Distribuição hipergeométrica - variância
def distribuicao_hipergeometrica_variancia(N: int, K: int, n: int) -> float:
    if N <= 1:
        raise ValueError("N deve ser maior que 1")
    return n * (K / N) * ((N - K) / N) * ((N - n) / (N - 1))


#Distribuição de Poisson
#Distribuição de Poisson - função de probabilidade
def distribuicao_poisson_pmf(lamb: float, k: int) -> float:
    if lamb < 0:
        raise ValueError("lambda deve ser não-negativo.")
    if k < 0:
        return 0.0
    return exponencial(-lamb) * (lamb ** k) / fatorial(k)


#Distribuição de Poisson - função de distribuição acumulada
def distribuicao_poisson_cdf(lamb: float, k: int) -> float:
    return sum(distribuicao_poisson_pmf(lamb, i) for i in range(0, k + 1))


#Distribuição de Poisson - média
def distribuicao_poisson_media(lamb: float) -> float:
    return lamb


#Distribuição de Poisson - variância
def distribuicao_poisson_variancia(lamb: float) -> float:
    return lamb


#Distribuição normal
#Distribuição normal - densidade de probabilidade
def distribuicao_normal_pdf(media: float, desvio: float, x: float) -> float:
    if desvio <= 0:
        raise ValueError("Desvio padrão deve ser positivo.")
    coeficiente = 1.0 / (desvio * sqrt(2.0 * pi))
    expoente = -((x - media) ** 2) / (2.0 * desvio * desvio)
    return coeficiente * exponencial(expoente)


#Distribuição normal - função de distribuição acumulada
def distribuicao_normal_cdf(media: float, desvio: float, x: float) -> float:
    if desvio <= 0:
        raise ValueError("Desvio padrão deve ser positivo.")
    z = (x - media) / (desvio * sqrt(2.0))
    return 0.5 * (1.0 + erf(z))


#Distribuição normal padrão - densidade de probabilidade
def distribuicao_normal_padrao_pdf(z: float) -> float:
    return distribuicao_normal_pdf(0.0, 1.0, z)


#Distribuição normal padrão - função de distribuição acumulada
def distribuicao_normal_padrao_cdf(z: float) -> float:
    return distribuicao_normal_cdf(0.0, 1.0, z)


#Distribuição exponencial
#Distribuição exponencial - densidade de probabilidade
def distribuicao_exponencial_pdf(taxa: float, x: float) -> float:
    if taxa <= 0:
        raise ValueError("Taxa deve ser positiva.")
    if x < 0:
        return 0.0
    return taxa * exponencial(-taxa * x)


#Distribuição exponencial - função de distribuição acumulada
def distribuicao_exponencial_cdf(taxa: float, x: float) -> float:
    if x < 0:
        return 0.0
    return 1.0 - exponencial(-taxa * x)


#Distribuição exponencial - média
def distribuicao_exponencial_media(taxa: float) -> float:
    return 1.0 / taxa


#Distribuição exponencial - variância
def distribuicao_exponencial_variancia(taxa: float) -> float:
    return 1.0 / (taxa * taxa)


#distribuição qui-quadrado
#Distribuição qui-quadrado - densidade de probabilidade
def distribuicao_qui_quadrado_pdf(graus_liberdade: int, x: float) -> float:
    if x < 0:
        return 0.0
    k = graus_liberdade
    numerador = (x ** (k / 2.0 - 1.0)) * exponencial(-x / 2.0)
    denominador = (2.0 ** (k / 2.0)) * gamma(k / 2.0)
    return numerador / denominador


#Distribuição qui-quadrado - média
def distribuicao_qui_quadrado_media(graus_liberdade: int) -> float:
    return float(graus_liberdade)


#Distribuição qui-quadrado - variância
def distribuicao_qui_quadrado_variancia(graus_liberdade: int) -> float:
    return 2.0 * graus_liberdade


#Distribuição t de Student
#Distribuição t de Student - densidade de probabilidade
def distribuicao_t_student_pdf(graus_liberdade: int, t: float) -> float:
    v = graus_liberdade
    numerador = gamma((v + 1) / 2.0)
    denominador = sqrt(v * pi) * gamma(v / 2.0)
    base = 1.0 + (t * t) / v
    return (numerador / denominador) * (base ** (-(v + 1) / 2.0))


#Amostragens pseudoaleatórias
#Amostragem pseudoaleatória uniforme
def amostragem_pseudoaleatoria_uniforme(quantidade: int, semente: int = 42) -> list:
    estado = semente
    modulo = 2 ** 31 - 1
    multiplicador = 1103515245
    incremento = 12345

    amostras = []
    for _ in range(quantidade):
        estado = (multiplicador * estado + incremento) % modulo
        amostras.append(estado / modulo)
    return amostras


#Amostragem pseudoaleatória normal
def amostragem_pseudoaleatoria_normal(quantidade: int, media: float = 0.0, desvio: float = 1.0, semente: int = 42) -> list:
    uniformes = amostragem_pseudoaleatoria_uniforme(quantidade * 2 + 2, semente)
    amostras = []
    i = 0
    while len(amostras) < quantidade:
        u1 = max(uniformes[i], 1e-12)
        u2 = uniformes[i + 1]
        i += 2
        raio = sqrt(-2.0 * ln(u1))
        z0 = raio * cos(2.0 * pi * u2)
        amostras.append(media + desvio * z0)
        if len(amostras) < quantidade:
            z1 = raio * sin(2.0 * pi * u2)
            amostras.append(media + desvio * z1)
    return amostras[:quantidade]


#Lei dos grandes números
def lei_dos_grandes_numeros_simulacao(funcao_amostra, n_amostras: int, semente: int = 42) -> float:
    uniformes = amostragem_pseudoaleatoria_uniforme(n_amostras, semente)
    soma = 0.0
    for u in uniformes:
        soma += funcao_amostra(u)
    return soma / n_amostras

Overwriting combinatoria_e_probabilidade.py
